### Notebook to demonstrate CLIP workflow

CLIP (Contrastive Language-Image Pre-Training) is a neural network trained on a variety of (image, text) pairs.
This notebook demonstrates how to fine-tune a CLIP model using the TAO SDK, covering training, evaluation, and export.

For CLIP dataset structure requirements, refer to the [TAO Toolkit documentation](https://docs.nvidia.com/tao/tao-toolkit).

**Actions supported:** `train`, `evaluate`, `export`, `inference`, `gen_trt_engine`

In [ ]:
import json
import os
import time
from IPython.display import clear_output

# Import TAO SDK
from tao_sdk.client import TaoClient

In [ ]:
# Restore variable in case of jupyter session restart and resume execution where it left off
%store -r model_name
%store -r base_url
%store -r headers
%store -r workspace_id
%store -r experiment_id
%store -r job_map

### FIXME's <a class="anchor" id="head-1"></a>

1. Assign model_name in FIXME 1
1. Set API host URL in FIXME 4
1. Set NGC Personal key in FIXME 5
1. Set NGC ORG name in FIXME 6
1. Set cloud workspace details in FIXME 7
1. Set dataset paths in FIXME 8

#### Choose a CLIP model

In [ ]:
# FIXME 1: Define model_name
# Available models:
# 1. clip - https://docs.nvidia.com/tao/tao-toolkit/text/cv_tasks/clip.html

os.environ["TAO_MODEL_NAME"] = model_name = os.environ.get("TAO_MODEL_NAME", "clip")
print(f"Model: {model_name}")

#### Set API service's host information

In [ ]:
# FIXME 4: Set TAO API environment variables

# Set to your TAO API endpoint
os.environ["TAO_BASE_URL"] = os.environ.get("TAO_BASE_URL", "http://localhost:8090/api/v2")
print(f"TAO API base URL: {os.environ['TAO_BASE_URL']}") 

#### Set NGC Personal key for authentication and NGC org to access API services

In [ ]:
# FIXME 5: Your NGC personal key
os.environ["NGC_KEY"] = ngc_key = os.environ.get("NGC_KEY", "your_ngc_key")

In [ ]:
# FIXME 6: Your NGC ORG name
os.environ["NGC_ORG"] = ngc_org_name = os.environ.get("NGC_ORG", "nvstaging")

### Login <a class="anchor" id="head-2"></a>

In [ ]:
# Initialize TAO Client and login using SDK
tao_client = TaoClient()

# Login using TAO SDK - this will automatically save credentials for subsequent calls
login_result = tao_client.login(ngc_key=ngc_key, ngc_org_name=ngc_org_name)
print(f"Logged in to org: {ngc_org_name}")

### Create cloud workspace
This workspace will be the place where your datasets reside and your results of TAO API jobs are stored. Each workspace is associated with one cloud bucket.

In [ ]:
# FIXME 7: Dataset Cloud bucket details to download dataset or push job artifacts for jobs

cloud_metadata = {}

# A Representative name for this cloud info
os.environ["TAO_WORKSPACE_NAME"] = cloud_metadata["name"] = os.environ.get("TAO_WORKSPACE_NAME", "AWS workspace info")

# Cloud specific details. Below is assuming AWS.
cloud_metadata["cloud_specific_details"] = {}

 # Whether it is AWS, HuggingFace or Azure
os.environ["TAO_WORKSPACE_CLOUD_TYPE"] = cloud_metadata["cloud_specific_details"]["cloud_type"] = os.environ.get("TAO_WORKSPACE_CLOUD_TYPE", "aws")

# Bucket region
os.environ["TAO_WORKSPACE_CLOUD_REGION"] = cloud_metadata["cloud_specific_details"]["cloud_region"] = os.environ.get("TAO_WORKSPACE_CLOUD_REGION", "us-west-1")

# Bucket name
os.environ["TAO_WORKSPACE_CLOUD_BUCKET_NAME"] = cloud_metadata["cloud_specific_details"]["cloud_bucket_name"] = os.environ.get("TAO_WORKSPACE_CLOUD_BUCKET_NAME", "bucket_name")

# Access and Secret keys
os.environ["TAO_WORKSPACE_CLOUD_ACCESS_KEY"] = cloud_metadata["cloud_specific_details"]["access_key"] = os.environ.get("TAO_WORKSPACE_CLOUD_ACCESS_KEY", "access_key")
os.environ["TAO_WORKSPACE_CLOUD_SECRET_KEY"] = cloud_metadata["cloud_specific_details"]["secret_key"] = os.environ.get("TAO_WORKSPACE_CLOUD_SECRET_KEY", "secret_key")

In [ ]:
# Create cloud workspace using TAO SDK
workspace_info = tao_client.create_workspace(
    name=cloud_metadata["name"],
    cloud_type=cloud_metadata["cloud_specific_details"]["cloud_type"],
    cloud_specific_details=cloud_metadata["cloud_specific_details"],
)

workspace_id = workspace_info["id"]
message = workspace_info.get("message", "")

if message:
    print("\u2139\ufe0f  " + message)
    print("=" * 80 + "\n")

print(f"Workspace created successfully!")
print(f"Workspace ID: {workspace_id}")

%store workspace_id

#### Set dataset URI (URI within cloud storage)

URIs should point to your dataset location in cloud storage, e.g. `aws://bucket-name/path/to/dataset`.

In [ ]:
# FIXME 8: Set paths relative to cloud bucket
os.environ["TAO_TRAIN_DATASET_URI"] = train_dataset_uri = os.environ.get("TAO_TRAIN_DATASET_URI", "/data/clip_train")
os.environ["TAO_EVAL_DATASET_URI"] = eval_dataset_uri = os.environ.get("TAO_EVAL_DATASET_URI", "/data/clip_val")

print(f"Train dataset path: {train_dataset_uri}")
print(f"Eval dataset path:  {eval_dataset_uri}")

### Set dataset format <a class="anchor" id="head-3"></a>

In [ ]:
# CLIP uses 'custom' dataset format
ds_format = "default"
print(f"Dataset format: {ds_format}")

### Set environment variables <a class="anchor" id="head-3.1"></a>

In [ ]:
# HuggingFace token is required to avoid rate limiting when downloading CLIP model weights
os.environ["HF_TOKEN"] = hf_token = os.environ.get("HF_TOKEN", "your_hf_token")  # FIXME: Add your HuggingFace token

docker_env_vars = {
    "HF_TOKEN": hf_token,
}

print(f"HF Token configured: {'yes' if hf_token != 'your_hf_token' else 'NOT SET - please fix'}")

### Set common params across all jobs <a class="anchor" id="head-8"></a>

In [ ]:
# These parameters are common to all jobs and will be used when creating the actual job:
encode_key = "nvidia_tao"
docker_env_vars = docker_env_vars if "docker_env_vars" in dir() else {}

### Actions <a class="anchor" id="head-13"></a>

For all actions:
1. Get default spec schema and derive the default values
1. Update the relevant spec values
1. Create the job
1. Monitor the job status

In [ ]:
job_map = {}
%store job_map

### Train <a class="anchor" id="head-14"></a>

In [ ]:
# Get default train specs using TAO SDK
train_spec_response = tao_client.get_job_schema(action="train", network_arch=model_name)
train_specs = train_spec_response["default"]
print(json.dumps(train_specs, sort_keys=True, indent=4))

In [ ]:
# Override train config parameters for CLIP
# Set number of epochs
if "train" in train_specs:
    train_specs["train"]["num_epochs"] = 1


print(json.dumps(train_specs, sort_keys=True, indent=4))

In [ ]:
# Create train job using TAO SDK
job_name = f"{model_name}_training_job"

job_info = tao_client.create_job(
    kind="experiment",
    name=job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="train",
    specs=train_specs,
    dataset_format=ds_format,
    train_dataset_uris=[train_dataset_uri],
    eval_dataset_uri=eval_dataset_uri,
    docker_env_vars=docker_env_vars,
)

job_id = job_info["id"]
message = job_info.get("message", "")

if message:
    print("\u2139\ufe0f  " + message)
    print("=" * 80 + "\n")
else:
    print("Train job created successfully!")
    print(f"Job ID: {job_id}")
    print(f"Job Name: {job_name}")
    print(f"Network Architecture: {model_name}")
    print(f"Action: train")

job_map["train_" + model_name] = job_id
print("\nJob Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
# Monitor training job status using TAO SDK
train_job_id = job_map["train_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(train_job_id)

        print(f"Training Job Status")
        print(f"Job ID: {train_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Training job failed!")

        if current_status in ["Done", "Completed"]:
            print("Training job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Training job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### Evaluate <a class="anchor" id="head-15"></a>

In [ ]:
# Get default eval specs using TAO SDK
eval_spec_response = tao_client.get_job_schema(action="evaluate", network_arch=model_name)
eval_specs = eval_spec_response["default"]
print(json.dumps(eval_specs, sort_keys=True, indent=4))

In [ ]:
# Override eval config parameters
print(json.dumps(eval_specs, sort_keys=True, indent=4))

In [ ]:
# Create evaluate job using TAO SDK
parent_job_id = job_map["train_" + model_name]
eval_job_name = f"{model_name}_eval_job"

eval_job_info = tao_client.create_job(
    kind="experiment",
    name=eval_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="evaluate",
    specs=eval_specs,
    dataset_format=ds_format,
    eval_dataset_uri=eval_dataset_uri,
    parent_job_id=parent_job_id,
    docker_env_vars=docker_env_vars,
)

eval_job_id = eval_job_info["id"]
message = eval_job_info.get("message", "")

if message:
    print("\u2139\ufe0f  " + message)
    print("=" * 80 + "\n")
else:
    print("Evaluate job created successfully!")
    print(f"Evaluate Job ID: {eval_job_id}")
    print(f"Parent Job ID: {parent_job_id}")
    print(f"Action: evaluate")

job_map["evaluate_" + model_name] = eval_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
# Monitor evaluate job status using TAO SDK
eval_job_id = job_map["evaluate_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(eval_job_id)

        print(f"Evaluate Job Status")
        print(f"Job ID: {eval_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Evaluate job failed!")

        if current_status in ["Done", "Completed"]:
            print("Evaluate job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Evaluate job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### Export <a class="anchor" id="head-17"></a>

In [ ]:
# Get default export specs using TAO SDK
export_spec_response = tao_client.get_job_schema(action="export", network_arch=model_name)
export_specs = export_spec_response["default"]
print(json.dumps(export_specs, sort_keys=True, indent=4))

In [ ]:
# Apply changes to export specs if required
# No dataset needed for export
print(json.dumps(export_specs, sort_keys=True, indent=4))

In [ ]:
# Create export job using TAO SDK
parent_job_id = job_map["train_" + model_name]
export_job_name = f"{model_name}_export_job"

export_job_info = tao_client.create_job(
    kind="experiment",
    name=export_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="export",
    specs=export_specs,
    parent_job_id=parent_job_id,
    docker_env_vars=docker_env_vars,
)

export_job_id = export_job_info["id"]
message = export_job_info.get("message", "")

if message:
    print("\u2139\ufe0f  " + message)
    print("=" * 80 + "\n")
else:
    print("Export job created successfully!")
    print(f"Export Job ID: {export_job_id}")
    print(f"Parent Job ID: {parent_job_id}")
    print(f"Action: export")

job_map["export_" + model_name] = export_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
# Monitor export job status using TAO SDK
export_job_id = job_map["export_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(export_job_id)

        print(f"Export Job Status")
        print(f"Job ID: {export_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Export job failed!")

        if current_status in ["Done", "Completed"]:
            print("Export job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Export job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### TAO Inference <a class="anchor" id="head-21"></a>

In [ ]:
# Get default inference specs using TAO SDK
inference_spec_response = tao_client.get_job_schema(action="inference", network_arch=model_name)
tao_inference_specs = inference_spec_response.get("default", {})
print("Default inference specifications:")
print(json.dumps(tao_inference_specs, sort_keys=True, indent=4))

In [ ]:
# Apply changes to tao_inference_specs if required
print(json.dumps(tao_inference_specs, sort_keys=True, indent=4))

In [ ]:
# Create inference job using TAO SDK
parent_job_id = job_map["train_" + model_name]
inference_job_name = f"{model_name}_inference_job"

inference_job_info = tao_client.create_job(
    kind="experiment",
    name=inference_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    docker_env_vars=docker_env_vars,
    train_dataset_uris=[train_dataset_uri],
    eval_dataset_uri=eval_dataset_uri,
    inference_dataset_uri=eval_dataset_uri,
    action="inference",
    specs=tao_inference_specs,  # Pass as dict, not JSON string
    dataset_format=ds_format,
    parent_job_id=parent_job_id,
)
inference_job_id = inference_job_info["id"]
message = inference_job_info.get("message", "")

if message:
    print("ℹ️  " + message)
    print("=" * 80 + "\n")
else:
    print("Inference job created successfully!")
    print(f"Inference Job ID: {inference_job_id}")
    print(f"Parent Job ID: {parent_job_id}")
    print(f"Action: inference")

job_map["inference_tao_" + model_name] = inference_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
inference_job_id = job_map["inference_tao_" + model_name]

while True:    
    clear_output(wait=True)
    
    try:
        job_status = tao_client.get_job_metadata(inference_job_id)
        
        print(f" Inference Job Status")
        print(f"Job ID: {inference_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")
        
        # Show detailed status information
        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))
        
        current_status = job_status.get("status", "Unknown")
        
        if current_status == "Error":
            raise Exception("Inference job failed!")
            
        if current_status in ["Done", "Completed"]:
            print("Inference job completed successfully!")
            break
            
        if current_status in ["Canceled", "Paused"]:
            print(f" Inference job {current_status}")
            break
            
    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f" Error fetching inference job status: {str(e)}")
        print("Job might still be starting up...")
        
    time.sleep(15)

### TRT Engine generation using TAO-Deploy <a class="anchor" id="head-20"></a>

- Here, we use the exported model to convert to target platform

In [ ]:
# Get default generate trt engine specs using TAO SDK
tao_deploy_spec_response = tao_client.get_job_schema(action="gen_trt_engine", network_arch=model_name)
tao_deploy_specs = tao_deploy_spec_response.get("default", {})
print("Default trt engine specifications:")
print(json.dumps(tao_deploy_specs, sort_keys=True, indent=4))

In [ ]:
# Apply changes to gen_trt_engine specs if necessary
print(json.dumps(tao_deploy_specs, sort_keys=True, indent=4))

In [ ]:
# Create generate trt engine job using TAO SDK
parent_job_id = job_map["export_" + model_name]
gen_trt_engine_job_name = f"{model_name}_gen_trt_engine_job"

gen_trt_engine_job_info = tao_client.create_job(
    kind="experiment",
    name=gen_trt_engine_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    action="gen_trt_engine",
    specs=tao_deploy_specs,  # Pass as dict, not JSON string
    parent_job_id=parent_job_id,
    docker_env_vars=docker_env_vars,
)
gen_trt_engine_job_id = gen_trt_engine_job_info["id"]
message = gen_trt_engine_job_info.get("message", "")

if message:
    print("ℹ️  " + message)
    print("=" * 80 + "\n")
else:
    print("Generate trt engine job created successfully!")
    print(f"Generate trt engine Job ID: {gen_trt_engine_job_id}")
    print(f"Parent Job ID: {parent_job_id}")
    print(f"Action: gen_trt_engine")

job_map["gen_trt_engine_" + model_name] = gen_trt_engine_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
# Monitor gen_trt_engine job status using TAO SDK
gen_trt_engine_job_id = job_map["gen_trt_engine_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(gen_trt_engine_job_id)

        print(f"Generate trt engine Job Status")
        print(f"Job ID: {gen_trt_engine_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("Generate trt engine job failed!")

        if current_status in ["Done", "Completed"]:
            print("Generate trt engine job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"Generate trt engine job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching gen_trt_engine job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### TRT Inference <a class="anchor" id="head-22"></a>

- No need to change the specs since we already set them at the TAO inference step

In [ ]:
# Get default trt inference specs using TAO SDK
trt_inference_spec_response = tao_client.get_job_schema(action="inference", network_arch=model_name)
trt_inference_specs = trt_inference_spec_response.get("default", {})
print("Default trt inference specifications:")
print(json.dumps(trt_inference_specs, sort_keys=True, indent=4))

In [ ]:
# Apply changes to trt inference specs if necessary
print(json.dumps(trt_inference_specs, sort_keys=True, indent=4))

In [ ]:
# Create trt inference job using TAO SDK
parent_job_id = job_map["gen_trt_engine_" + model_name]
trt_inference_job_name = f"{model_name}_trt_inference_job"

trt_inference_job_info = tao_client.create_job(
    kind="experiment",
    name=trt_inference_job_name,
    network_arch=model_name,
    encryption_key=encode_key,
    workspace=workspace_id,
    inference_dataset_uri=eval_dataset_uri,
    action="inference",
    specs=trt_inference_specs,  # Pass as dict, not JSON string
    parent_job_id=parent_job_id,
    docker_env_vars=docker_env_vars,
)
trt_inference_job_id = trt_inference_job_info["id"]
message = trt_inference_job_info.get("message", "")

if message:
    print("ℹ️  " + message)
    print("=" * 80 + "\n")
else:
    print("TRT Inference job created successfully!")
    print(f"TRT Inference Job ID: {trt_inference_job_id}")
    print(f"Parent Job ID: {parent_job_id}")
    print(f"Action: inference")

job_map["trt_inference_" + model_name] = trt_inference_job_id
print("\nUpdated Job Map:")
print(json.dumps(job_map, indent=4))
%store job_map

In [ ]:
# Monitor trt inference job status using TAO SDK
trt_inference_job_id = job_map["trt_inference_" + model_name]

while True:
    clear_output(wait=True)

    try:
        job_status = tao_client.get_job_metadata(trt_inference_job_id)

        print(f"TRT Inference Job Status")
        print(f"Job ID: {trt_inference_job_id}")
        print(f"Status: {job_status.get('status', 'Unknown')}")
        print(f"Progress: {job_status.get('progress', 'N/A')}")

        print("\nDetailed Status:")
        print(json.dumps(job_status.get("job_details", {}), sort_keys=True, indent=4))

        current_status = job_status.get("status", "Unknown")

        if current_status == "Error":
            raise Exception("TRT Inference job failed!")

        if current_status in ["Done", "Completed"]:
            print("TRT Inference job completed successfully!")
            break

        if current_status in ["Canceled", "Paused"]:
            print(f"TRT Inference job {current_status}")
            break

    except Exception as e:
        if "failed" in str(e).lower():
            raise
        print(f"Error fetching trt inference job status: {str(e)}")
        print("Job might still be starting up...")

    time.sleep(15)

### Delete Jobs<a class="anchor" id="head-22"></a>

In [ ]:
# Delete all created jobs using TAO SDK

print("Deleting all created jobs...")

for job_key, job_id in job_map.items():
    try:
        delete_result = tao_client.delete_job(job_id)
        print(f"Deleted job: {job_key} (ID: {job_id})")
    except Exception as e:
        print(f"Failed to delete job {job_key} (ID: {job_id}): {str(e)}")

print("All jobs deleted.")